# Run TMB_RFD2 inference

This notebook is a beginner-friendly example for running **TMB_RFD2** after you have:

cloned the `TMB_pipeline` repository



## 1. Imports


In [10]:
import os, shutil, sys

print("cwd:", os.getcwd())
print("python:", sys.executable)
print("git:", shutil.which("git"))
print("curl:", shutil.which("curl"))
print("apptainer:", shutil.which("apptainer"))
print("singularity:", shutil.which("singularity"))

cwd: /mnt/home/aphilo/TMB_pipeline/TMB_RFD2
python: /opt/conda/envs/env/bin/python
git: None
curl: /opt/conda/envs/env//bin/curl
apptainer: None
singularity: None


In [11]:
import os
from pathlib import Path

# go to your repo
repo = Path("/mnt/home/aphilo/TMB_pipeline/TMB_RFD2")
os.chdir(repo)
print("cwd:", os.getcwd())

# set container path
APPTAINER_PATH = "/home/aphilo/sif/beta_barrel.sif"   # change if needed
os.environ["APPTAINER_PATH"] = APPTAINER_PATH

# bind useful paths
os.environ["APPTAINER_BINDPATH"] = "/mnt/home/aphilo,/home/aphilo,/projects"

# check runtime
!which apptainer || which singularity
!apptainer --version || singularity --version

# test the container
!apptainer exec "$APPTAINER_PATH" cat /etc/os-release || singularity exec "$APPTAINER_PATH" cat /etc/os-release

cwd: /mnt/home/aphilo/TMB_pipeline/TMB_RFD2
/bin/bash: line 1: apptainer: command not found
/bin/bash: line 1: singularity: command not found
/bin/bash: line 1: apptainer: command not found
/bin/bash: line 1: singularity: command not found


In [16]:
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
ENV_DIR.mkdir(parents=True, exist_ok=True)

print("Directories created.")

Directories created.


In [20]:
import shutil, os

APPTAINER_BIN = shutil.which("apptainer") or shutil.which("singularity")

if APPTAINER_BIN is None:
    raise RuntimeError("Apptainer/Singularity not found. Install it on the system or load the module.")

os.environ["APPTAINER_BIN"] = APPTAINER_BIN

print("Using:", APPTAINER_BIN)

RuntimeError: Apptainer/Singularity not found. Install it on the system or load the module.

In [21]:
!module load apptainer || module load singularity
!which apptainer || which singularity

/bin/bash: line 1: module: command not found
/bin/bash: line 1: module: command not found


## 2. Read environment variables

Make sure you already ran the setup commands from the README in your shell before starting Jupyter.


In [2]:
def first_existing(paths):
    for p in paths:
        if p and Path(p).exists():
            return str(Path(p).resolve())
    return None

home = Path.home()
cwd = Path.cwd()

# Start with environment variables
RFD2_CKPT_PATH = os.environ.get("RFD2_CKPT_PATH")
APPTAINER_PATH = os.environ.get("APPTAINER_PATH")
TMB_PIPELINE_DIR = os.environ.get("TMB_PIPELINE_DIR", str(cwd))

# Try common fallback locations
APPTAINER_PATH = first_existing([
    APPTAINER_PATH,
    home / "ENV_DIR" / "modulus_latest.sif",
    cwd / "modulus_latest.sif",
    cwd.parent / "modulus_latest.sif",
])

RFD2_CKPT_PATH = first_existing([
    RFD2_CKPT_PATH,
    home / "WEIGHTS_DIR" / "RFD_TMBB_bulges_twists_masked_withPDB__10.pt",
    cwd / "RFD_TMBB_bulges_twists_masked_withPDB__10.pt",
    cwd.parent / "RFD_TMBB_bulges_twists_masked_withPDB__10.pt",
])

TMB_PIPELINE_DIR = Path(TMB_PIPELINE_DIR).resolve()

assert APPTAINER_PATH is not None, (
    "Could not find the container file.\n"
    "Set APPTAINER_PATH in your environment, or place modulus_latest.sif in a common location such as:\n"
    f"  {home / 'sif' / 'modulus_latest.sif'}\n"
    f"  {cwd / 'modulus_latest.sif'}"
)

assert RFD2_CKPT_PATH is not None, (
    "Could not find the RFD2 checkpoint.\n"
    "Set RFD2_CKPT_PATH in your environment, or place the checkpoint in a common location such as:\n"
    f"  {home / 'WEIGHTS_DIR' / 'RFD_TMBB_bulges_twists_masked_withPDB__10.pt'}\n"
    f"  {cwd / 'RFD_TMBB_bulges_twists_masked_withPDB__10.pt'}"
)

RFD2_CKPT_PATH = Path(RFD2_CKPT_PATH).resolve()
APPTAINER_PATH = Path(APPTAINER_PATH).resolve()

print("APPTAINER_PATH:", APPTAINER_PATH)
print("RFD2_CKPT_PATH:", RFD2_CKPT_PATH)
print("TMB_PIPELINE_DIR:", TMB_PIPELINE_DIR)

APPTAINER_PATH: /home/aphilo/ENV_DIR/modulus_latest.sif
RFD2_CKPT_PATH: /home/aphilo/WEIGHTS_DIR/RFD_TMBB_bulges_twists_masked_withPDB__10.pt
TMB_PIPELINE_DIR: /mnt/home/aphilo/TMB_pipeline/TMB_RFD2


## 3. User-editable settings

Edit these if your repo structure or example input locations are different.


In [14]:
# -----------------------------
# USER SETTINGS
# -----------------------------

# Repository code locations
RFD2_CODE_DIR = TMB_PIPELINE_DIR / "TMB_RFD2"
ADJ_SS_WRAPPER_SCRIPT = TMB_PIPELINE_DIR / "scripts" / "adj_ss_wrapper_glykink.py"

# Update this to a real example PDB inside your repo
INPUT_PDB = TMB_PIPELINE_DIR / "example_inputs" / "input.pdb"

# RFD2 settings
CONFIG_NAME = "aa_ppi"
HAS_TERMINI = "[True]"
CONTIGS = "['1-1']"
CENTER_TYPE = "all"
NUM_DESIGNS = 10

# Example design condition
N_STRANDS = 12
RESIDUES_PER_STRAND = 16
GLYCINE_TWISTS = "1 2 7 8"

# Output locations
WORK_DIR = TMB_PIPELINE_DIR / "notebook_runs"
ADJ_SS_DIR = WORK_DIR / "adj_ss"
OUTPUT_DIR = WORK_DIR / "backbones"
COMMANDS_FILE = WORK_DIR / "inference_commands.txt"

WORK_DIR.mkdir(parents=True, exist_ok=True)
ADJ_SS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Work directory:", WORK_DIR)


Work directory: /home/aphilo/TMB_pipeline/TMB_RFD2/notebook_runs


In [3]:
# -----------------------------
# REBUILD COMMAND FILE WITH CORRECT PATH
# -----------------------------
from pathlib import Path

# Since notebook is already inside /home/aphilo/TMB_pipeline/TMB_RFD2
RFD2_CODE_DIR = Path(".").resolve()

adj_files = sorted(ADJ_SS_DIR.glob("*adj.pt"))
ss_files = sorted(ADJ_SS_DIR.glob("*ss.pt"))
twist_files = sorted(ADJ_SS_DIR.glob("*twist.pt"))

assert adj_files, f"No adjacency file found in {ADJ_SS_DIR}"
assert ss_files, f"No secondary structure file found in {ADJ_SS_DIR}"

ADJ_FILE = adj_files[0]
SS_FILE = ss_files[0]
TWIST_FILE = twist_files[0] if twist_files else None

print("Using RFD2_CODE_DIR:", RFD2_CODE_DIR)
print("ADJ_FILE:", ADJ_FILE)
print("SS_FILE:", SS_FILE)
if TWIST_FILE:
    print("TWIST_FILE:", TWIST_FILE)

# Change the script name here to the real inference entrypoint in this repo
INFERENCE_SCRIPT = RFD2_CODE_DIR / "run_inference.py"   # <-- replace if needed

inference_cmd = (
    f"python {INFERENCE_SCRIPT} "
    f"--config-name {CONFIG_NAME} "
    f"inference.output_prefix={OUTPUT_DIR}/design "
    f"inference.input_pdb={INPUT_PDB} "
    f"inference.num_designs={NUM_DESIGNS} "
    f"contigmap.contigs=\"{CONTIGS}\" "
    f"inference.has_termini=\"{HAS_TERMINI}\" "
    f"ppi.center_type={CENTER_TYPE} "
    f"scaffoldguided.target_adj={ADJ_FILE} "
    f"scaffoldguided.target_ss={SS_FILE}"
)

if TWIST_FILE is not None:
    inference_cmd += f" scaffoldguided.target_twist={TWIST_FILE}"

COMMANDS_FILE.write_text(inference_cmd + "\n")

print(f"\nCreated command file: {COMMANDS_FILE}")
print("\nContents:\n")
print(COMMANDS_FILE.read_text())

NameError: name 'ADJ_SS_DIR' is not defined

In [5]:
!python /home/aphilo/TMB_pipeline/TMB_RFD2/run_inference.py --config-name aa_ppi inference.output_prefix=/home/aphilo/TMB_pipeline/TMB_RFD2/notebook_runs/backbones/design inference.input_pdb=/home/aphilo/TMB_pipeline/TMB_RFD2/example_inputs/input.pdb inference.num_designs=10 contigmap.contigs="['1-1']" inference.has_termini="[True]" ppi.center_type=all scaffoldguided.target_adj=/home/aphilo/TMB_pipeline/TMB_RFD2/notebook_runs/adj_ss/n12_nres16_adj.pt scaffoldguided.target_ss=/home/aphilo/TMB_pipeline/TMB_RFD2/notebook_runs/adj_ss/n12_nres16_ss.pt scaffoldguided.target_twist=/home/aphilo/TMB_pipeline/TMB_RFD2/notebook_runs/adj_ss/n12_nres16_twist.pt

python: can't open file '/home/aphilo/TMB_pipeline/TMB_RFD2/run_inference.py': [Errno 2] No such file or directory


In [8]:
!apptainer exec /home/aphilo/ENV_DIR/modulus_latest.sif !python /home/aphilo/TMB_pipeline/TMB_RFD2/TMB_RFD2_code/rf_diffusion/run_inference.py --config-name aa_ppi inference.output_prefix=/home/aphilo/TMB_pipeline/TMB_RFD2/notebook_runs/backbones/design inference.input_pdb=/home/aphilo/TMB_pipeline/TMB_RFD2/example_inputs/input.pdb inference.num_designs=10 contigmap.contigs="['1-1']" inference.has_termini="[True]" ppi.center_type=all scaffoldguided.target_adj=/home/aphilo/TMB_pipeline/TMB_RFD2/notebook_runs/adj_ss/n12_nres16_adj.pt scaffoldguided.target_ss=/home/aphilo/TMB_pipeline/TMB_RFD2/notebook_runs/adj_ss/n12_nres16_ss.pt scaffoldguided.target_twist=/home/aphilo/TMB_pipeline/TMB_RFD2/notebook_runs/adj_ss/n12_nres16_twist.pt

/bin/bash: line 1: apptainer: command not found
